# Modelagem da Camada Gold: Fato Vendas

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.services.spark_session import get_spark_session, close_spark_session
import src.modules.modeling_fato_utils as modeling_fato
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("ModelagemGoldFatoVendas")

# Leitura das tabelas de origem

In [ ]:
# Define caminhos das origens na Silver e Gold
silver_vendas_path = "s3a://silver/vendas"
gold_dim_local_path = "s3a://gold/dim_local"

# Lê os dados definindo como None caso a origem não exista
try:
    df_vendas = spark.read.parquet(silver_vendas_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Vendas não encontrada: {e}")
    df_vendas = None

try:
    df_dim_local = spark.read.parquet(gold_dim_local_path)
except Exception as e:
    print(f"Aviso: Tabela Gold dim_local não encontrada: {e}")
    df_dim_local = None

# Visualização Prévia dos Dados de Entrada

In [ ]:
if df_vendas is not None:
    print("=== Colunas em Silver Vendas ===")
    display(df_vendas.limit(5).toPandas())

if df_dim_local is not None:
    print("=== Colunas em Gold dim_local ===")
    display(df_dim_local.limit(5).toPandas())

# Cria a Fato Vendas (fato_vendas)

In [ ]:
# Executa a lógica de modelagem da fato
df_fato_vendas = modeling_fato.create_fato_vendas(df_vendas, df_dim_local)

if df_fato_vendas is not None:
    # Exibe informações sobre o DataFrame gerado
    print(f"Quantidade total de registros na fato vendas: {df_fato_vendas.count()}")
    df_fato_vendas.printSchema()
    display(df_fato_vendas.limit(10).toPandas())
else:
    print("Nenhuma fato de vendas foi processada devido a ausência da origem de vendas.")

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)